In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
pip install -q -U transformers accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [3]:
# milestone_qwen_zeroshot.py
# zero-shot MCQ scoring using Qwen3, single forward-pass per question (no generation)
# scores options by grabbing next-token logits right after "Answer:" instead of decoding text

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"   # swap to Qwen/Qwen3-4B if you're short on time
BATCH_SIZE = 8
MAX_LEN = 512

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")  # fix path to your actual dataset

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"  # so last position lines up across a padded batch
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

LETTERS = ["A", "B", "C", "D", "E"]

# figure out which token id corresponds to each bare letter for this tokenizer
# (some BPE vocabs tokenize " A" as one piece, others split it — check both)
letter_ids = {}
for l in LETTERS:
    cand = tok.encode(l, add_special_tokens=False)
    cand_space = tok.encode(" " + l, add_special_tokens=False)
    # pick whichever encodes to a single token, prefer the space-prefixed one since
    # that's what actually shows up after "Answer:"
    if len(cand_space) == 1:
        letter_ids[l] = cand_space[0]
    elif len(cand) == 1:
        letter_ids[l] = cand[0]
    else:
        # fallback: just take the last sub-token, rare edge case
        letter_ids[l] = cand_space[-1]

def build_prompt(row):
    opts = "\n".join(f"{l}) {row[l]}" for l in LETTERS)
    user_msg = (
        f"Question: {row['prompt']}\n\n{opts}\n\n"
        "Pick the single best answer. Reply with only the letter."
    )
    msgs = [{"role": "user", "content": user_msg}]
    text = tok.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # skip the <think> block, we only need next-token logits
    )
    text += "Answer:"
    return text

prompts = [build_prompt(r) for _, r in test_df.iterrows()]
letter_id_list = [letter_ids[l] for l in LETTERS]

all_preds = []
with torch.no_grad():
    for i in range(0, len(prompts), BATCH_SIZE):
        batch = prompts[i:i + BATCH_SIZE]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                   max_length=MAX_LEN).to(model.device)
        out = model(**enc)
        last_logits = out.logits[:, -1, :]  # left-padded, so -1 is always the real last token
        option_logits = last_logits[:, letter_id_list]  # (batch, 5)
        probs = torch.softmax(option_logits, dim=-1).float().cpu().numpy()

        for row_probs in probs:
            ranked = np.argsort(row_probs)[::-1][:3]
            top3 = " ".join(LETTERS[j] for j in ranked)
            all_preds.append(top3)

        if i % 80 == 0:
            print(f"done {i}/{len(prompts)}")

sub = pd.DataFrame({"ID": test_df["id"], "Prediction": all_preds})
sub.to_csv("submission.csv", index=False)
print(sub.head())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

done 0/500
done 80/500
done 160/500
done 240/500
done 320/500
done 400/500
done 480/500
   ID Prediction
0   1      A D B
1   2      B D A
2   3      B A C
3   4      E C D
4   5      C A E
